# Topic: Design an Enterprise RAG Search System

## Definition (30-second explanation)
*   **Analogy:** Imagine a massive corporate library. You cannot just let any employee walk in and read the CEO’s private diary. Furthermore, truckloads of new books arrive every minute, and the librarians need a system to catalog them without shutting down the checkout desk. 
*   An Enterprise RAG (Retrieval-Augmented Generation) system is a highly scalable architecture that ingests millions of company documents, converts them to vector embeddings asynchronously, and retrieves them for LLMs while strictly enforcing data privacy and user access permissions.

## Why Interviewers Ask This
*   Anyone can build a local RAG app in a Jupyter notebook using LangChain and a PDF. Interviewers ask this to see if you understand the absolute nightmare of deploying RAG in a Fortune 500 company: **Data Privacy (RBAC), Scalability, and Multi-Tenancy.**

## Core Concepts (The Enterprise Architecture)
1.  **The Ingestion Pipeline (Data Connectors):** 
    *   Enterprise data lives everywhere (Confluence, Jira, Google Drive, PDFs). You need a scheduled cron job or event-driven pipeline (e.g., Apache Airflow) to crawl these sources, extract text, and chunk it properly.
2.  **Async Embedding Generation:** 
    *   If you try to embed 100,000 pages of a new API documentation synchronously, your system will crash, or you will hit OpenAI's rate limits. 
    *   *Solution:* The ingestion pipeline drops text chunks into a queue (like Kafka or AWS SQS). Background worker nodes pull from the queue, generate embeddings at a controlled rate, and write them to the Vector DB.
3.  **Vector DB Scaling & Multi-Tenant Isolation:** 
    *   **Multi-Tenancy:** If you are a B2B SaaS company, Client A's data must NEVER touch Client B's data. You achieve this either by using separate physical databases (expensive) or by using **Namespaces/Collections** within a single Vector DB (like Pinecone or Milvus) to logically isolate tenants.
4.  **Access Control (RBAC - Role-Based Access Control):** 
    *   This is the most critical enterprise feature. A junior engineer cannot ask the LLM, *"What is the salary of the VP of Engineering?"* and get an answer from HR's private documents.
    *   *Solution:* During ingestion, every vector is tagged with metadata (e.g., `role_allowed: ["HR", "Executive"]`). When the user queries the system, the API checks their JWT token, extracts their role, and applies a **Pre-Filter** to the Vector DB search so it *only* searches vectors they are allowed to see.
5.  **Latency SLAs:** 
    *   LLM generation already takes 2-5 seconds. The retrieval part (Vector Search) must be lightning fast (<100ms), or the user experience feels broken.

## Common Interview Traps
*   **Ignoring Metadata Filtering:** Suggesting that you will just retrieve the top 10 documents and have the *LLM* decide if the user is allowed to read them. *Correction:* The LLM should never even see unauthorized documents. The Vector DB must filter them out before they reach the prompt.
*   **Synchronous Processing:** Designing an architecture where uploading a 500-page PDF blocks the UI while it waits for embeddings to generate.

## 45-Second Interview Answer
"To design an Enterprise RAG system, I break it into two asynchronous pipelines. The ingestion pipeline crawls enterprise sources, chunks the text, and uses a message queue like Kafka to asynchronously generate embeddings without hitting rate limits. Crucially, every embedded chunk is tagged with RBAC metadata. On the retrieval side, when a user queries the system, the API decodes their authentication token to identify their role and tenant ID. It passes these as a strict pre-filter to the Vector DB, ensuring the cosine similarity search only executes over documents they have explicit permission to view, keeping retrieval under 100 milliseconds before passing context to the LLM."

## Practice Questions:

### Q1: Enterprise RAG & RBAC
**Question:** 
How do you architect a RAG system so an "Associate" only searches public files, but a "Partner" can search public and confidential files, without breaking the vector database's latency?

**Answer:**
"To ensure data security without destroying search latency, I rely on **Role-Based Access Control (RBAC) combined with Single-Stage Vector Filtering**. 
During the ingestion pipeline, every document chunk is embedded and tagged with metadata (e.g., `access_level: public` or `access_level: confidential`). 
When an Associate queries the UI, the API decodes their JWT token, identifies their role, and passes a strict metadata filter (`allowed: public`) directly into the vector database search query. 
Crucially, I would use a modern vector database like Pinecone or Qdrant that supports **In-Search (Single-Stage) Filtering**. Instead of naively pre-filtering (which isolates nodes and breaks HNSW graphs), the engine dynamically evaluates the metadata permissions *during* the graph traversal, ensuring we maintain sub-100ms latency while guaranteeing the Associate never retrieves a Partner's confidential documents."

**Common Interview Traps:**
* Not knowing that strict pre-filtering breaks HNSW graphs (the "Island Effect").
* Assuming the LLM handles security. *Correction: The LLM is highly susceptible to prompt injection; security MUST be handled at the Vector DB level before the prompt is ever created.*